# BERT-style

Devlin et al., *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*, NAACL 2019 ([arXiv:1810.04805](https://arxiv.org/abs/1810.04805)).

Encoder-only, bidirectional self-attention (no mask), learned positional embeddings, masked-language-modeling objective (the exact 80/10/10 recipe). Trained here on real WikiText-2.

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from transformer_playground.device import resolve_device
from transformer_playground.utils.seed import set_seed
from model import BERTModel
from example import Vocab, MLMDataset, _load_lines

set_seed(0)
device = resolve_device('auto')
print(device)

In [ ]:
train_lines = _load_lines('train')[:3000]
val_lines = _load_lines('valid')[:300]
vocab = Vocab([t for line in train_lines for t in line])
max_len = 32
train_ds = MLMDataset(train_lines, vocab, max_len)
val_ds = MLMDataset(val_lines, vocab, max_len)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
print(len(train_ds), len(val_ds), len(vocab))

In [ ]:
model = BERTModel(vocab_size=len(vocab), max_len=max_len, d_model=128, n_heads=4, n_layers=2, d_ff=256).to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

history = {'train_loss': []}
for epoch in range(10):
    model.train()
    total = 0.0
    for input_ids, labels in train_loader:
        input_ids, labels = input_ids.to(device), labels.to(device)
        opt.zero_grad()
        logits = model(input_ids)
        loss = loss_fn(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1))
        loss.backward()
        opt.step()
        total += loss.item()
    history['train_loss'].append(total / len(train_loader))
    print(epoch, history['train_loss'][-1])

In [ ]:
plt.plot(history['train_loss'])
plt.xlabel('epoch')
plt.ylabel('MLM train loss')
plt.title('BERT-style masked-LM training loss (real WikiText-2)')
plt.show()